In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import pandas as pd
df = pd.read_csv("release_train_patients.csv")  # try CSV
print(df.shape)
print(df.columns.tolist())

# ── Load dataset ───────────────────────────────────────────────────────────
#file_path = "release_train_patients.xlsx"
#df = pd.read_excel(file_path) if file_path.endswith(".xlsx") else pd.read_csv(file_path)

required = ["AGE", "SEX", "PATHOLOGY", "EVIDENCES", "DIFFERENTIAL_DIAGNOSIS"]
for c in required:
    if c not in df.columns:
        raise ValueError(f"Column '{c}' not found. Available: {list(df.columns)}")

df_clean = df.dropna(subset=required).copy()

# ── Derived stats ──────────────────────────────────────────────────────────
evidence_counts   = df_clean["EVIDENCES"].astype(str).apply(lambda x: len(x.split(",")))
pathology_counts  = df_clean["PATHOLOGY"].value_counts()
missing_counts    = df.isnull().sum()

# ── Summary print ──────────────────────────────────────────────────────────
print("\n==============================")
print("DATA ANALYSIS (SUMMARY)")
print("==============================")
print(f"Observations (rows):  {df.shape[0]:,}")
print(f"Features (columns):   {df.shape[1]}")
print(f"Complete rows:        {df_clean.shape[0]:,}")

print("\n--- Unique counts ---")
print(f"Unique pathologies:   {df_clean['PATHOLOGY'].nunique()}")
print(f"Sex categories:       {df_clean['SEX'].unique().tolist()}")

print("\n--- Age statistics ---")
print("Mean:",   round(df_clean["AGE"].mean(), 2))
print("Std:",    round(df_clean["AGE"].std(), 2))
print("Min:",    int(df_clean["AGE"].min()))
print("Q1:",     round(df_clean["AGE"].quantile(0.25), 2))
print("Median:", round(df_clean["AGE"].median(), 2))
print("Q3:",     round(df_clean["AGE"].quantile(0.75), 2))
print("Max:",    int(df_clean["AGE"].max()))

print("\n--- Evidence count per patient ---")
print("Mean:",   round(evidence_counts.mean(), 2))
print("Std:",    round(evidence_counts.std(), 2))
print("Min:",    int(evidence_counts.min()))
print("Q1:",     round(evidence_counts.quantile(0.25), 2))
print("Median:", round(evidence_counts.median(), 2))
print("Q3:",     round(evidence_counts.quantile(0.75), 2))
print("Max:",    int(evidence_counts.max()))

print("\n--- Missing values (all columns) ---")
print(missing_counts)

# ── Table 1: Pathology distribution ───────────────────────────────────────
path_table = pd.DataFrame({
    "Count":   pathology_counts,
    "Percent": (pathology_counts / pathology_counts.sum() * 100).round(2)
})
print("\n==============================")
print("Table 1: Distribution of Pathologies (top 15)")
print("==============================")
print(path_table.head(15).to_string())

# ── Table 2: Evidence count statistics ────────────────────────────────────
table2 = pd.DataFrame({
    "Avg":          [evidence_counts.mean()],
    "Std dev":      [evidence_counts.std()],
    "Min":          [evidence_counts.min()],
    "1st quartile": [evidence_counts.quantile(0.25)],
    "Median":       [evidence_counts.median()],
    "3rd quartile": [evidence_counts.quantile(0.75)],
    "Max":          [evidence_counts.max()]
}).round(2)
print("\n==============================")
print("Table 2: Statistics of evidence count per patient")
print("==============================")
print(table2.to_string(index=False))

# ── Sample row ─────────────────────────────────────────────────────────────
print("\n==============================")
print("Sample row (example patient)")
print("==============================")
print(df_clean.sample(1).to_string(index=False))


# ── Figure 1: Top 10 pathologies bar chart ─────────────────────────────────
plt.figure()
pathology_counts.head(10).plot(kind="bar", color="#1e3a5f")
plt.title("Figure 1: Top 10 Pathologies")
plt.xlabel("Pathology")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# ── Figure 2: Sex distribution bar chart ──────────────────────────────────
plt.figure()
df_clean["SEX"].value_counts().plot(kind="bar", color=["#1e3a5f", "#4a9fd4"])
plt.title("Figure 2: Sex Distribution")
plt.xlabel("Sex")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# ── Figure 3: Age histogram ────────────────────────────────────────────────
plt.figure()
plt.hist(df_clean["AGE"], bins=20, color="#1e3a5f", edgecolor="white")
plt.axvline(df_clean["AGE"].mean(), color="red", linestyle="--",
            label=f"Mean = {df_clean['AGE'].mean():.1f}")
plt.title("Figure 3: Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of Patients")
plt.legend()
plt.tight_layout()
plt.show()

# ── Figure 4: Evidence count histogram ────────────────────────────────────
plt.figure()
plt.hist(evidence_counts, bins=20, color="#4a9fd4", edgecolor="white")
plt.axvline(evidence_counts.mean(), color="red", linestyle="--",
            label=f"Mean = {evidence_counts.mean():.1f}")
plt.title("Figure 4: Distribution of Evidence Count per Patient")
plt.xlabel("Number of Evidences")
plt.ylabel("Number of Patients")
plt.legend()
plt.tight_layout()
plt.show()

# ── Figure 5: Pathology pie chart (top 8 + Other) ─────────────────────────
plt.figure()
top8  = pathology_counts.head(8)
other = pathology_counts.iloc[8:].sum()
pie_data = pd.concat([top8, pd.Series({"Other": other})]) if other > 0 else top8
plt.pie(pie_data.values, labels=pie_data.index, autopct="%1.1f%%",
        startangle=90, textprops={"fontsize": 8})
plt.title("Figure 5: Pathology Share (Top 8 + Other)")
plt.tight_layout()
plt.show()

# ── Figure 6: Missing values bar chart ────────────────────────────────────
plt.figure()
missing_counts.plot(kind="bar", color="#b0ddf0", edgecolor="#1e3a5f")
plt.title("Figure 6: Missing Values per Column")
plt.xlabel("Column")
plt.ylabel("Missing Count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()